In [1]:
import pandas as pd
import sqlite3
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# ── 1. DATOS DE EJEMPLO (simula el input del challenge) ──────────
raw_data = {
    'order_id':       ['O001','O002','O003','O004','O005'],
    'customer_name':  ['Ana García','Ana García','Bob López','Bob López','Carlos M.'],
    'customer_email': ['ana@mail.com','ana@mail.com','bob@mail.com','bob@mail.com','carlos@mail.com'],
    'customer_city':  ['CDMX','CDMX','GDL','GDL','MTY'],
    'product_name':   ['Laptop','Mouse','Desk','Chair','Laptop'],
    'category':       ['Tech','Tech','Furniture','Furniture','Tech'],
    'unit_price':     [1200.0, 25.0, 300.0, 150.0, 1200.0],
    'qty':            [1, 2, 1, 4, 1],
    'order_date':     ['2024-01-15','2024-01-15','2024-02-01','2024-02-01','2024-03-10'],
    'status':         ['completed','completed','completed','pending','completed']
}
df_raw = pd.DataFrame(raw_data)
df_raw['total'] = df_raw['unit_price'] * df_raw['qty']

# ── 2. NORMALIZACIÓN ─────────────────────────────────────────────
def create_schema(conn):
    conn.executescript('''
        DROP TABLE IF EXISTS order_items;
        DROP TABLE IF EXISTS orders;
        DROP TABLE IF EXISTS products;
        DROP TABLE IF EXISTS customers;

        CREATE TABLE customers (
            customer_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            name          TEXT NOT NULL,
            email         TEXT UNIQUE NOT NULL,
            city          TEXT
        );
        CREATE TABLE products (
            product_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            product_name  TEXT NOT NULL,
            category      TEXT,
            unit_price    REAL CHECK(unit_price > 0),
            UNIQUE(product_name, category)
        );
        CREATE TABLE orders (
            order_id      TEXT PRIMARY KEY,
            customer_id   INTEGER NOT NULL,
            order_date    DATE,
            status        TEXT DEFAULT 'pending',
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
        CREATE TABLE order_items (
            item_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id      TEXT NOT NULL,
            product_id    INTEGER NOT NULL,
            qty           INTEGER CHECK(qty > 0),
            unit_price    REAL,
            FOREIGN KEY (order_id)    REFERENCES orders(order_id),
            FOREIGN KEY (product_id)  REFERENCES products(product_id)
        );
    ''')
    logging.info("Schema creado")

def load_normalized(df: pd.DataFrame, conn) -> None:
    cur = conn.cursor()

    # Clientes únicos
    customers = df[['customer_name','customer_email','customer_city']]\
        .drop_duplicates('customer_email')
    for _, row in customers.iterrows():
        cur.execute(
            "INSERT OR IGNORE INTO customers(name,email,city) VALUES(?,?,?)",
            (row.customer_name, row.customer_email, row.customer_city)
        )

    # Productos únicos
    products = df[['product_name','category','unit_price']]\
        .drop_duplicates(['product_name','category'])
    for _, row in products.iterrows():
        cur.execute(
            "INSERT OR IGNORE INTO products(product_name,category,unit_price) VALUES(?,?,?)",
            (row.product_name, row.category, row.unit_price)
        )

    # Órdenes y líneas de detalle
    for _, row in df.iterrows():
        cust_id = cur.execute(
            "SELECT customer_id FROM customers WHERE email=?",
            (row.customer_email,)
        ).fetchone()[0]

        prod_id = cur.execute(
            "SELECT product_id FROM products WHERE product_name=? AND category=?",
            (row.product_name, row.category)
        ).fetchone()[0]

        cur.execute(
            "INSERT OR IGNORE INTO orders(order_id,customer_id,order_date,status) VALUES(?,?,?,?)",
            (row.order_id, cust_id, row.order_date, row.status)
        )
        cur.execute(
            "INSERT INTO order_items(order_id,product_id,qty,unit_price) VALUES(?,?,?,?)",
            (row.order_id, prod_id, row.qty, row.unit_price)
        )

    conn.commit()
    logging.info("Datos cargados correctamente")

# ── 3. VERIFICACIÓN con queries ──────────────────────────────────
def verify(conn) -> None:
    checks = {
        'customers': 'SELECT * FROM customers',
        'products':  'SELECT * FROM products',
        'orders':    'SELECT * FROM orders',
        'items':     'SELECT * FROM order_items',
        'resumen por cliente': '''
            SELECT c.name, COUNT(DISTINCT o.order_id) AS orders,
                   SUM(i.qty * i.unit_price) AS total
            FROM customers c
            JOIN orders o ON c.customer_id = o.customer_id
            JOIN order_items i ON o.order_id = i.order_id
            GROUP BY c.customer_id ORDER BY total DESC
        '''
    }
    for name, query in checks.items():
        df = pd.read_sql(query, conn)
        print(f"\n── {name} ──")
        print(df.to_string(index=False))

# ── 4. EJECUTAR ──────────────────────────────────────────────────
with sqlite3.connect(':memory:') as conn:  # :memory: = DB en RAM
    create_schema(conn)
    load_normalized(df_raw, conn)
    verify(conn)
    logging.info("Challenge completado")

INFO: Schema creado
INFO: Datos cargados correctamente
INFO: Challenge completado



── customers ──
 customer_id       name           email city
           1 Ana García    ana@mail.com CDMX
           2  Bob López    bob@mail.com  GDL
           3  Carlos M. carlos@mail.com  MTY

── products ──
 product_id product_name  category  unit_price
          1       Laptop      Tech      1200.0
          2        Mouse      Tech        25.0
          3         Desk Furniture       300.0
          4        Chair Furniture       150.0

── orders ──
order_id  customer_id order_date    status
    O001            1 2024-01-15 completed
    O002            1 2024-01-15 completed
    O003            2 2024-02-01 completed
    O004            2 2024-02-01   pending
    O005            3 2024-03-10 completed

── items ──
 item_id order_id  product_id  qty  unit_price
       1     O001           1    1      1200.0
       2     O002           2    2        25.0
       3     O003           3    1       300.0
       4     O004           4    4       150.0
       5     O005           1   